# 3_clustering_analysis
Apply k-Means clustering to TF-IDF vectors and analyze cluster centroids.

In [2]:
%%bash
pip install -q scikit-learn pandas matplotlib seaborn joblib


In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import joblib
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import os
print('imports ok')

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports ok


In [2]:
dataset = load_dataset('hpe-ai/medical-cases-classification-tutorial')
train_df = pd.DataFrame(dataset['train'])
X = train_df['transcription'].fillna('')
tfidf = joblib.load('artifacts/tfidf_vectorizer.joblib') if os.path.exists('artifacts/tfidf_vectorizer.joblib') else None
if tfidf is None:
    raise RuntimeError('Please run notebook 1 to fit and save the TF-IDF vectorizer to artifacts/tfidf_vectorizer.joblib')
X_vec = tfidf.transform(X)
print('Vectorized shape:', X_vec.shape)


Repo card metadata block was not found. Setting CardData to empty.


Vectorized shape: (1724, 104781)


## Determine k with Silhouette score
Run KMeans for a range of k and inspect silhouette scores.

In [3]:
sil_scores = {}
for k in range(2,9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_vec)
    score = silhouette_score(X_vec, labels)
    sil_scores[k] = score
sil_scores


{2: 0.010607603840288174,
 3: 0.011948886043353875,
 4: 0.013356939826835544,
 5: 0.014674353683329202,
 6: 0.014594625335241711,
 7: 0.018037643464968313,
 8: 0.01795687896040578}

In [5]:
# Fit KMeans with chosen k (example chooses k=5)
k = 5
km = KMeans(n_clusters=k, random_state=42, n_init=10)
km.fit(X_vec)
labels = km.labels_
train_df['cluster'] = labels
train_df.groupby('cluster')['medical_specialty'].value_counts().head(20)


cluster  medical_specialty         
0        Cardiovascular / Pulmonary    132
         Radiology                       3
         Neurology                       2
         Gastroenterology                1
         Neurosurgery                    1
1        Orthopedic                    165
         Cardiovascular / Pulmonary    156
         Gastroenterology               76
         Obstetrics / Gynecology        67
         Neurosurgery                   44
         ENT - Otolaryngology           38
         Ophthalmology                  32
         Hematology - Oncology          20
         Neurology                      13
         Nephrology                     12
         Radiology                       4
2        Cardiovascular / Pulmonary    158
         Hematology - Oncology          61
         Gastroenterology               58
         Neurology                      57
Name: count, dtype: int64

In [6]:
# Inspect top terms per cluster (centroid)
terms = tfidf.get_feature_names_out()
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
for i in range(k):
    top_terms = [terms[ind] for ind in order_centroids[i, :20]]
    print(f'Cluster {i} top terms: ', ', '.join(top_terms[:15]))


Cluster 0 top terms:  artery, coronary, left, right, aortic, coronary artery, valve, catheter, stenosis, right coronary, french, vessel, descending, ventricular, circumflex
Cluster 1 top terms:  patient, procedure, placed, right, left, incision, anesthesia, removed, using, used, diagnosis, taken, closed, performed, tube
Cluster 2 top terms:  history, patient, mg, pain, past, normal, daily, denies, does, medications, medical, day, disease, heart, family
Cluster 3 top terms:  c5, c6, c4, c5 c6, cervical, c3, c7, c4 c5, c6 c7, c3 c4, anterior, anterior cervical, plate, discectomy, mm
Cluster 4 top terms:  right, normal, unremarkable, left, exam, fetal, mri, revealed, ct, spine, seen, patient, evidence, mild, pain


In [1]:
import joblib

# Load trained model
pipeline = joblib.load("artifacts/model_pipeline.joblib")

# Example input
sample_text = "changes in movement such as tremors, weakness, or coordination problems, and sensory issues like numbness or tingling"

# Predict
prediction = pipeline.predict([sample_text])
print("Predicted Specialty:", prediction[0])

Predicted Specialty: Neurology


## Business insights
Write observations about how clusters map to specialties, whether clusters identify sub-topics, and how MedArchive might use them.